# Item-KNN Content-Based - Proposal Compliant

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
# Keep linear algebra backends single-threaded to reduce noise in timing
# and to improve reproducibility across machines.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

# Global random seed used by all stochastic steps in this notebook.
np.random.seed(42)

## Preprocessing and evaluation protocol

In [2]:
from pathlib import Path

# -----------------------------
# 1) Load raw input tables
# -----------------------------
# All baselines share the same CSV inputs so results are comparable.
DATA_DIR = Path("data")
MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_PATH = DATA_DIR / "ratings.csv"
USERS_PATH = DATA_DIR / "user_profiles.csv"

movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)
user_profiles_df = pd.read_csv(USERS_PATH)

# -----------------------------
# 2) Standardize schemas
# -----------------------------
# Rename columns to a unified format expected by all methods.
ratings = ratings_df.rename(columns={"id": "user_id", "rate": "rating"}).copy()
movies = movies_df.copy()
users = user_profiles_df.rename(columns={"id": "user_id"}).copy()

# Keep only columns needed in this notebook and remove obvious invalid rows.
ratings = ratings[["user_id", "movie_id", "rating"]].dropna()
movies = movies.drop_duplicates(subset=["id"]).copy()
users = users.drop_duplicates(subset=["user_id"]).copy()

# Keep only user flags that actually exist in the current dataset.
flag_cols = [c for c in ["early_bird", "night_owl", "weekend_tweeter"] if c in users.columns]
if "followers_count" not in users.columns:
    raise ValueError("Expected a followers_count column in user_profiles.csv")

# Ensure optional item metadata columns always exist.
for col in ["genres", "actors", "directors"]:
    if col not in movies.columns:
        movies[col] = ""

# Fill missing metadata to prevent tokenization/feature errors.
movies["genres"] = movies["genres"].fillna("")
movies["actors"] = movies["actors"].fillna("")
movies["directors"] = movies["directors"].fillna("")
movies["year_published"] = pd.to_numeric(movies.get("year_published"), errors="coerce")


def split_tokens(value):
    # Parse pipe-separated fields like "Action|Drama" into token lists.
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]


def split_train_test_by_user(ratings_df, train_ratio=0.80, seed=42):
    # Split each user's interactions into train/test.
    # This keeps user identities in both splits and removes validation set.
    rng = np.random.default_rng(seed)
    train_parts = []
    test_parts = []

    for _, group in ratings_df.groupby("user_id"):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)

        n_rows = len(idx)
        if n_rows <= 1:
            train_idx = idx
            test_idx = np.array([], dtype=idx.dtype)
        else:
            n_train = max(1, int(np.floor(n_rows * train_ratio)))
            if n_train >= n_rows:
                n_train = n_rows - 1
            train_idx = idx[:n_train]
            test_idx = idx[n_train:]

        train_parts.append(ratings_df.loc[train_idx])
        if len(test_idx) > 0:
            test_parts.append(ratings_df.loc[test_idx])

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=ratings_df.columns)
    test_df = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame(columns=ratings_df.columns)
    return train_df, test_df


train_df, test_df = split_train_test_by_user(ratings, train_ratio=0.80, seed=42)

# Cached list of all genre tokens (useful for consistency checks/extensions).
all_genres = sorted({g for s in movies["genres"].fillna("") for g in split_tokens(s)})
all_item_set = set(movies["id"].dropna().astype(int).tolist())


def build_user_pos_lookup(df):
    # Build dictionary: user_id -> set(positive movie_ids).
    lookup = {}
    for uid, group in df.groupby("user_id"):
        lookup[int(uid)] = set(group["movie_id"].astype(int).tolist())
    return lookup


def merge_user_pos_lookups(*lookups):
    # Merge multiple user->set(item) maps into one map.
    merged = {}
    for lookup in lookups:
        for uid, items in lookup.items():
            if uid not in merged:
                merged[uid] = set()
            merged[uid].update(items)
    return merged


train_user_pos = build_user_pos_lookup(train_df)
test_user_pos = build_user_pos_lookup(test_df)
all_user_pos = merge_user_pos_lookups(train_user_pos, test_user_pos)


def sample_negatives(candidate_pool, exclude_set, n_negatives=20, rng=None):
    # Sample negative candidate items while excluding known positives.
    # With replacement is enabled only when pool is smaller than request.
    rng = rng or np.random.default_rng(42)
    pool = np.array([i for i in candidate_pool if i not in exclude_set], dtype=int)
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < n_negatives
    return rng.choice(pool, size=n_negatives, replace=replace)


def ranking_metrics_from_rank(rank, k=10):
    # Convert a positive item's rank into ranking metrics.
    # Recall@K is binary per example in this one-positive setup.
    recall = 1.0 if rank <= k else 0.0
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0
    mrr = 1.0 / rank
    return recall, ndcg, mrr


def rank_positive_among_candidates(scores, positive_mask, rng=None):
    # Tie-safe rank: if multiple candidates share the same score as positive,
    # randomize only within the tie group to avoid order bias.
    rng = rng or np.random.default_rng(42)
    positive_idx = int(np.flatnonzero(positive_mask)[0])
    pos_score = float(scores[positive_idx])

    better = int(np.sum(scores > pos_score))
    tie_indices = np.flatnonzero(scores == pos_score)
    tie_count = int(len(tie_indices))

    if tie_count <= 1:
        return better + 1, tie_count

    shuffled_ties = tie_indices[rng.permutation(tie_count)]
    tie_pos = int(np.where(shuffled_ties == positive_idx)[0][0])
    return better + tie_pos + 1, tie_count


def evaluate_ranker(score_fn, positives_df, candidate_pool, all_user_pos_lookup, n_negatives=20, k=10, seed=42):
    # Evaluate a scoring function in sampled ranking setting:
    # one positive + N negatives per interaction.
    rng = np.random.default_rng(seed)
    rows = []
    tie_cases = 0

    for uid, group in positives_df.groupby("user_id"):
        uid = int(uid)
        known_positives = all_user_pos_lookup.get(uid, set())

        for _, row in group.iterrows():
            pos_item = int(row["movie_id"])
            exclude = set(known_positives)
            exclude.add(pos_item)

            negatives = sample_negatives(candidate_pool, exclude, n_negatives=n_negatives, rng=rng)
            candidate_items = np.array([pos_item] + negatives.tolist(), dtype=int)
            positive_mask = np.zeros(len(candidate_items), dtype=bool)
            positive_mask[0] = True

            # Shuffle candidate order so metric does not depend on [positive + negatives] layout.
            perm = rng.permutation(len(candidate_items))
            candidate_items = candidate_items[perm]
            positive_mask = positive_mask[perm]

            scores = score_fn(uid, candidate_items)
            rank, tie_count = rank_positive_among_candidates(scores, positive_mask, rng=rng)
            if tie_count > 1:
                tie_cases += 1

            recall, ndcg, mrr = ranking_metrics_from_rank(rank, k=k)
            rows.append((recall, ndcg, mrr))

    if not rows:
        return {"Recall@10": np.nan, "NDCG@10": np.nan, "MRR": np.nan, "n_eval": 0, "tie_rate": np.nan}

    arr = np.array(rows, dtype=float)
    n_eval = int(len(rows))
    return {
        "Recall@10": float(arr[:, 0].mean()),
        "NDCG@10": float(arr[:, 1].mean()),
        "MRR": float(arr[:, 2].mean()),
        "n_eval": n_eval,
        "tie_rate": float(tie_cases / n_eval),
    }


print(f"Movies: {movies.shape[0]:,} | Ratings: {ratings.shape[0]:,} | Users: {users.shape[0]:,}")
print(f"Split rows -> train: {len(train_df):,}, test: {len(test_df):,}")
print(f"Users in split -> train: {train_df['user_id'].nunique():,}, test: {test_df['user_id'].nunique():,}")

Movies: 78,628 | Ratings: 1,172,038 | Users: 482
Split rows -> train: 937,385, test: 234,653
Users in split -> train: 611, test: 611


## Method-specific training and evaluation

In [3]:
from sklearn.feature_extraction import FeatureHasher, DictVectorizer
from sklearn.preprocessing import normalize

# -------------------------------------------
# 3) Build ItemKNN content representations
# -------------------------------------------

def build_item_feature_dicts(movies_df, hash_dim=128):
    # Build sparse item feature dicts from metadata.
    # Actors/directors are hashed to keep dimensionality compact.
    movies_local = movies_df.copy()
    year_min = movies_local["year_published"].min(skipna=True)
    year_max = movies_local["year_published"].max(skipna=True)
    year_span = (year_max - year_min) if pd.notna(year_min) and pd.notna(year_max) and year_max != year_min else 1.0
    movies_local["year_norm"] = ((movies_local["year_published"] - year_min) / year_span).fillna(0.0).clip(0.0, 1.0)

    hasher = FeatureHasher(n_features=hash_dim, input_type="string", alternate_sign=False)
    content_dicts = []
    for _, row in movies_local.iterrows():
        genres = split_tokens(row.get("genres", ""))
        actors = split_tokens(row.get("actors", ""))
        directors = split_tokens(row.get("directors", ""))

        feat = {}
        for g in genres:
            feat[f"genre__{g}"] = 1.0
        feat["year_norm"] = float(row["year_norm"])

        tokens = [f"actor::{a}" for a in actors] + [f"director::{d}" for d in directors]
        if tokens:
            hashed = hasher.transform([tokens]).toarray().ravel()
            for i, val in enumerate(hashed):
                if val != 0:
                    feat[f"hash__{i}"] = float(val)

        content_dicts.append((int(row["id"]), feat))

    return content_dicts


def dicts_to_sparse_matrix(dict_items):
    # Vectorize dictionaries into L2-normalized sparse item matrix.
    ids = [k for k, _ in dict_items]
    vec = DictVectorizer(sparse=True)
    X = vec.fit_transform([d for _, d in dict_items])
    return ids, normalize(X, norm="l2", copy=False)


def build_user_profiles_from_content(X_items, train_df, item_row_map):
    # Build one profile vector per user as weighted sum of rated item vectors.
    # Ratings are used as weights to emphasize stronger preferences.
    user_profiles = {}
    for uid, group in train_df.groupby("user_id"):
        uid = int(uid)
        idxs = []
        weights = []
        for _, row in group.iterrows():
            item = int(row["movie_id"])
            if item in item_row_map:
                idxs.append(item_row_map[item])
                weights.append(float(row["rating"]))
        if not idxs:
            continue

        vec = X_items[idxs].T @ np.array(weights, dtype=float)
        vec = np.asarray(vec).ravel()
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        user_profiles[uid] = vec
    return user_profiles


# Build content matrix and user profile vectors from training interactions.
item_content = build_item_feature_dicts(movies, hash_dim=128)
item_ids, X_item = dicts_to_sparse_matrix(item_content)
item_row_map = {mid: i for i, mid in enumerate(item_ids)}
user_profiles = build_user_profiles_from_content(X_item, train_df, item_row_map)


def knn_score_fn(uid, candidate_items):
    # Score candidates by cosine-like similarity between user profile and item.
    # Unseen users/items receive neutral zero score.
    user_vec = user_profiles.get(uid)
    if user_vec is None:
        return np.zeros(len(candidate_items), dtype=float)

    scores = []
    for item in candidate_items:
        if item in item_row_map:
            row_idx = item_row_map[item]
            scores.append(float(X_item[row_idx].dot(user_vec.T).sum()))
        else:
            scores.append(0.0)
    return np.array(scores, dtype=float)


# -------------------------------------------
# 4) Evaluate on test split
# -------------------------------------------
knn_test_metrics = evaluate_ranker(
    knn_score_fn,
    test_df,
    all_item_set,
    all_user_pos,
    n_negatives=20,
    k=10,
    seed=43,
 )

print("Item-KNN Content-Based")
print("Test:", knn_test_metrics)

Item-KNN Content-Based
Test: {'Recall@10': 0.6494270262898834, 'NDCG@10': 0.3213023865389511, 'MRR': 0.2470512194805245, 'n_eval': 234653, 'tie_rate': 4.261611826825142e-06}
